# CUDA generator verification only
The notebook runs Step A and preserves its evidence.

In [ ]:
REPO_URL = "https://github.com/AdebanjiAdelowo/physicsnemo-fno-darcy.git"
REF = "1f8a69e8a7c0772bf95bab4ed9022f1a7c8510d1"   # output of `git rev-parse HEAD` for the commit to run
RUN_FULL = False                              # step B: only the value True starts it
RUN_OFFICIAL = False                          # step C: only the value True starts it
SESSION_BUDGET_HOURS = 9.0                    # GPU time you are willing to spend in this session; check Kaggle's current limits
GITHUB_TOKEN_SECRET = None                    # name of a Kaggle secret holding a GitHub token; only for a private repository

import re, time
T0 = time.time()
assert re.fullmatch(r"[0-9a-f]{40}", REF), "REF must be a full 40-character commit SHA, not a branch or tag."
WORK = "/kaggle/working"
REPO = "/tmp/physicsnemo-fno-darcy"

def hours_left():
    return SESSION_BUDGET_HOURS - (time.time() - T0) / 3600

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "CUDA is not available. Select a GPU accelerator; this notebook does not fall back to CPU."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
import os, subprocess
def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

if not os.path.isdir(REPO):
    url = REPO_URL
    if GITHUB_TOKEN_SECRET:
        from kaggle_secrets import UserSecretsClient
        url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET) + "@")
    subprocess.run(["git", "clone", "--quiet", url, REPO], check=True)   # not echoed: the URL may hold a token
    subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--quiet", REF], cwd=REPO, check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True).stdout.strip()
assert head == REF, f"checked out {head}, expected {REF}"
assert not dirty, "the working tree is not clean"
print("commit", head)

Install the pinned PhysicsNeMo commit and the CPU audit versions of PyTorch, Warp, NumPy and SciPy into a fresh virtual environment. The next check reports the installed versions and confirms CUDA availability.

In [ ]:
import sys
VENV = "/tmp/darcy-verification-env"
subprocess.run([sys.executable, "-m", "venv", "--without-pip", VENV], check=True)
PYTHON = f"{VENV}/bin/python"
subprocess.run([sys.executable, "-m", "pip", "--python", PYTHON, "install", "-q", "pip"], check=True)
os.environ["PATH"] = f"{VENV}/bin:" + os.environ["PATH"]
# Isolate from Kaggle's optional RAPIDS stack. Match the CPU audit's Warp and
# numerical libraries to make the cross-device comparison easier to interpret.
sh("python -m pip install -q -r requirements.txt torch==2.14.1 torchvision==0.29.1 "
   "warp-lang==1.18.0 numpy==2.5.3 scipy==1.18.1", cwd=REPO)
sh('python -c "import torch, physicsnemo, warp; assert torch.cuda.is_available(); '
   "print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| physicsnemo', physicsnemo.__version__, '| warp', warp.__version__, '|', torch.cuda.get_device_name(0))\"", cwd=REPO)

## Step A: generator checks on CUDA

Runs before any training. It measures, with the Warp CUDA backend, (1) how many distinct permeability fields a batch of `Darcy2D` holds, as shipped and with `IndependentDarcy2D`, and (2) the distance of the generated pressure from sparse direct solves of the kernel stencil and of the conservative Darcy equation. The GPU and the CUDA, Warp, PyTorch and PhysicsNeMo versions are stored in the JSON file.

In [ ]:
import json, zipfile
CHECK = f"{WORK}/generator-check-cuda"   # outside the clone, so the tree stays clean for the studies
from pathlib import Path
evidence = Path(CHECK)
evidence.mkdir(exist_ok=True)
(evidence / "nvidia-smi.txt").write_text(subprocess.check_output(["nvidia-smi"], text=True))
(evidence / "pip-freeze.txt").write_text(subprocess.check_output([PYTHON, "-m", "pip", "freeze"], text=True))
command = [PYTHON, "-u", "scripts/verify_generator.py", "--device", "cuda", "--out-root", CHECK]
with (evidence / "verification.log").open("w") as log:
    with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, env={**os.environ, "MPLBACKEND": "Agg"}) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
        status = proc.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)
check = json.load(open(f"{CHECK}/results/generator_check_cuda.json"))
env = check["environment"]
print({k: env[k] for k in ("gpu_name", "cuda_version", "torch_version", "warp_version", "physicsnemo")})
shipped = check["diversity"]["Darcy2D"]
repeated = max(shipped["distinct_fields_per_batch"]) == 1
print(f"Darcy2D as shipped on CUDA: distinct fields per batch of {shipped['batch_size']}: {shipped['distinct_fields_per_batch']}, "
      f"distinct coefficient values: {shipped['distinct_coefficient_values_per_batch']} of {shipped['coefficients_per_batch']}")
print("REPEATED-BATCH DEFECT ON CUDA:", "REPRODUCED" if repeated else "NOT REPRODUCED (compare with results/generator_check.json, CPU)")
with zipfile.ZipFile(f"{WORK}/physicsnemo-fno-darcy-generator-check-cuda.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for rel in ("results/generator_check_cuda.json", "figures/generator_check_cuda.png", "verification.log", "nvidia-smi.txt", "pip-freeze.txt"):
        z.write(f"{CHECK}/{rel}", rel)
print("written:", f"{WORK}/physicsnemo-fno-darcy-generator-check-cuda.zip")
assert not subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True).stdout.strip(), "step A modified the clone"
print(f"Elapsed including notebook setup: {time.time() - T0:.1f} s")
